In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("NYC_Taxi_Stage4_SQL").getOrCreate()
sc = spark.sparkContext
sc.setLogLevel("ERROR")

print("Loading clean data...")
# 1. Load the optimized Parquet from Stage 3
clean_df = spark.read.parquet("../data/processed/clean_taxis.parquet")

# 2. RUBRIC: Register the DataFrame as a temporary view
clean_df.createOrReplaceTempView("taxis")
print("Temporary view 'taxis' registered successfully.\n")

# --- QUERY 1: Class Balance (Gold Requirement) ---
print("QUERY 1: Class balance and payment method")
query_1 = spark.sql("""
    SELECT 
        payment_type,
        CASE WHEN payment_type = 1 THEN 'Credit Card' 
             WHEN payment_type = 2 THEN 'Cash' 
             ELSE 'Other' END AS payment_name,
        COUNT(*) as total_trips,
        SUM(CASE WHEN tip_amount > 0 THEN 1 ELSE 0 END) as trips_with_tip,
        ROUND(SUM(CASE WHEN tip_amount > 0 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as tip_percentage
    FROM taxis
    GROUP BY payment_type, payment_name
    ORDER BY total_trips DESC
""")
query_1.show()

# --- QUERY 2: Temporal Trends (CTEs and Window Functions) ---
print("QUERY 2: Most generous hours (Window Functions)")
query_2 = spark.sql("""
    WITH HourlyStats AS (
        SELECT 
            HOUR(CAST(tpep_pickup_datetime AS TIMESTAMP)) as hour_of_day,
            COUNT(*) as total_trips,
            ROUND(AVG(CASE WHEN tip_amount > 0 THEN 1 ELSE 0 END) * 100, 2) as tip_probability,
            ROUND(AVG(tip_amount), 2) as avg_tip_dollars
        FROM taxis
        WHERE payment_type = 1 -- Filter only cards to see real tips
        GROUP BY HOUR(CAST(tpep_pickup_datetime AS TIMESTAMP))
    )
    SELECT 
        hour_of_day, 
        total_trips, 
        tip_probability,
        avg_tip_dollars,
        RANK() OVER (ORDER BY avg_tip_dollars DESC) as generosity_rank
    FROM HourlyStats
    ORDER BY generosity_rank
    LIMIT 5
""")
query_2.show()

# --- QUERY 3: Rate Code Segmentation (HAVING and Grouping) ---
print("QUERY 3: Standard trips vs. Airports")
query_3 = spark.sql("""
    SELECT 
        RateCodeID,
        CASE WHEN RateCodeID = 1 THEN 'Standard'
             WHEN RateCodeID = 2 THEN 'JFK Airport'
             WHEN RateCodeID = 3 THEN 'Newark Airport'
             ELSE 'Other' END as rate_type,
        COUNT(*) as total_trips,
        ROUND(AVG(fare_amount), 2) as avg_fare,
        ROUND(AVG(tip_amount), 2) as avg_tip
    FROM taxis
    WHERE payment_type = 1
    GROUP BY RateCodeID, rate_type
    HAVING COUNT(*) > 10000 -- Exclude rare categories with few trips
    ORDER BY avg_tip DESC
""")
query_3.show()

# --- QUERY 4: Distance vs Tip Distribution (CASE conditionals) ---
print("QUERY 4: Does distance guarantee a tip?")
query_4 = spark.sql("""
    SELECT 
        CASE 
            WHEN trip_distance < 2 THEN '1. Short (<2m)'
            WHEN trip_distance BETWEEN 2 AND 5 THEN '2. Medium (2-5m)'
            ELSE '3. Long (>5m)'
        END as distance_category,
        COUNT(*) as total_trips,
        ROUND(AVG(CASE WHEN tip_amount > 0 THEN 1 ELSE 0 END) * 100, 2) as probability_of_tip,
        ROUND(AVG(tip_amount), 2) as avg_tip_dollars
    FROM taxis
    WHERE payment_type = 1
    GROUP BY 
        CASE 
            WHEN trip_distance < 2 THEN '1. Short (<2m)'
            WHEN trip_distance BETWEEN 2 AND 5 THEN '2. Medium (2-5m)'
            ELSE '3. Long (>5m)'
        END
    ORDER BY distance_category
""")
query_4.show()

# --- QUERY 5: Toll Impact (Multivariable numeric correlation) ---
print("QUERY 5: Toll correlation with tips")
query_5 = spark.sql("""
    SELECT 
        CASE WHEN tolls_amount > 0 THEN 'Paid Tolls' ELSE 'No Tolls' END as toll_status,
        COUNT(*) as total_trips,
        ROUND(AVG(fare_amount), 2) as avg_fare,
        ROUND(AVG(tip_amount), 2) as avg_tip,
        ROUND(AVG(tip_amount) / AVG(fare_amount) * 100, 2) as tip_to_fare_ratio
    FROM taxis
    WHERE payment_type = 1
    GROUP BY CASE WHEN tolls_amount > 0 THEN 'Paid Tolls' ELSE 'No Tolls' END
""")
query_5.show()

26/05/27 02:13:41 WARN Utils: Your hostname, LaptopPablo resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/05/27 02:13:41 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/05/27 02:13:41 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Loading clean data...


Temporary view 'taxis' registered successfully.

QUERY 1: Class balance and payment method


...

+----------+---------+-----------+--------+-------+
|RateCodeID|rate_type|total_trips|avg_fare|avg_tip|
+----------+---------+-----------+--------+-------+
|         1| Standard|    7265052|   10.44|   2.14|
+----------+---------+-----------+--------+-------+

QUERY 4: Does distance guarantee a tip?



[Stage 13:===>                                                    (1 + 15) / 16]



+-----------------+-----------+------------------+---------------+
|distance_category|total_trips|probability_of_tip|avg_tip_dollars|
+-----------------+-----------+------------------+---------------+
|   1. Short (<2m)|    4309090|             96.46|            1.6|
| 2. Medium (2-5m)|    2423924|             96.76|           2.64|
|    3. Long (>5m)|     533186|             95.53|           4.23|
+-----------------+-----------+------------------+---------------+

QUERY 5: Toll correlation with tips


+-----------+-----------+--------+-------+-----------------+
|toll_status|total_trips|avg_fare|avg_tip|tip_to_fare_ratio|
+-----------+-----------+--------+-------+-----------------+
|   No Tolls|    7189019|   10.31|   2.11|            20.45|
| Paid Tolls|      77181|   22.08|   5.42|            24.54|
+-----------+-----------+--------+-------+-----------------+




[Stage 16:=================>                                      (5 + 11) / 16]

